## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from gemini_vertex import openai_client, MODEL_ID
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Gemini on Vertex AI needs no API key - it uses your Google login (Application Default Credentials).
# Check the project and model that are set in the .env file

project = os.getenv('GOOGLE_CLOUD_PROJECT')

if project:
    print(f"Using Google Cloud project {project} (location: {os.getenv('GOOGLE_CLOUD_LOCATION')})")
    print(f"Model from .env: {os.getenv('GEMINI_MODEL')}")
else:
    print("GOOGLE_CLOUD_PROJECT not set - check the .env file in the project root")


In [ ]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = openai_client()

response = openai.chat.completions.create(model=MODEL_ID, messages=messages)
question = response.choices[0].message.content
display(Markdown(question))


## Getting several answers to compare

The original lab calls LLMs from many different providers. Here we use just one model: the Gemini model named in the `.env` file (`GEMINI_MODEL`).

To keep the same pattern - several competitors, then an LLM as a judge - we ask that model for three independent answers and let it judge them.


In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [ ]:
# Ask the model from .env for three independent answers

for attempt in range(1, 4):
    response = openai.chat.completions.create(model=MODEL_ID, messages=messages)
    answer = response.choices[0].message.content

    record(f"{MODEL_ID} (attempt {attempt})", answer)


In [ ]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

## And now the judge

The same model from `.env` acts as our LLM as a judge

In [ ]:
# Judgement time!

response = openai.chat.completions.create(model=MODEL_ID, messages=judge_messages)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!
# Gemini sometimes wraps the JSON in a code block, or writes "competitor 2" instead of "2" - so we clean up first

import re

results_dict = json.loads(re.sub(r"```(json)?", "", results).strip())
ranks = [int(re.search(r"\d+", str(r)).group()) for r in results_dict["results"]]
for index, result in enumerate(ranks):
    competitor = competitors[result-1]
    print(f"Rank {index+1}: {competitor}")


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>